# Tahoe LFC Heldout Molecules

Each point is the mean L2 across 45 Tahoe cell lines. Error bars are 95% bootstrap confidence intervals across cell lines.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

repo = Path.cwd()
while not (repo / "results" / "scores" / "tahoe_lfc_heldout_molecules_fresh.csv").exists() and repo != repo.parent:
    repo = repo.parent
score_path = repo / "results" / "scores" / "tahoe_lfc_heldout_molecules_fresh.csv"
fig_dir = repo / "results" / "figures" / "fig_tahoe_lfc_heldout_molecules"
fig_dir.mkdir(parents=True, exist_ok=True)
with open(repo / "results" / "metadata" / "fig_index.json") as f:
    fig_index = json.load(f)

mpl.rcParams.update(fig_index.get("mpl_params", {}))
palette = {**fig_index["drugs_model_type_palette"], "Response Embedding": "#4C78A8"}
label_map = {
    "morgan_initialized_lpm": ("Morgan-initialized LPM", "Response Embedding"),
    "random": ("Random Embeddings", "Negative Control"),
    "pca": ("PCA / idealized baseline", "Positive Control"),
    "ChemBERTa-77M-MLM": ("ChemBERTa-77M-MLM", "SMILES Transformer"),
    "ChemBERTa-77M-MTR": ("ChemBERTa-77M-MTR", "SMILES Transformer"),
    "chatgpt": ("ChatGPT", "LLM"),
    "maccs": ("MACCS", "Molecule Structure"),
    "topological": ("Topological", "Molecule Structure"),
    "secfp": ("SECFP", "Molecule Structure"),
    "avalon": ("Avalon", "Molecule Structure"),
    "erg": ("ErG", "Molecule Structure"),
    "ecfp:2": ("ECFP:2", "Molecule Structure"),
    "MiniMol": ("MiniMol", "Molecule Structure"),
    "MolT5": ("MolT5", "SMILES Transformer"),
    "unimol2-570m-H": ("Uni-Mol2 570M-H", "Molecule Structure"),
    "boltz_affinity_pred_value_fragment": ("Boltz affinity fragment", "Protein Affinity"),
    "boltz_affinity_pred_value_protein": ("Boltz affinity protein", "Protein Affinity"),
}

df = pd.read_csv(score_path)
df["Display name"] = df["embedding"].map(lambda x: label_map.get(x, (x, "Molecule Structure"))[0])
df["Model type"] = df["embedding"].map(lambda x: label_map.get(x, (x, "Molecule Structure"))[1])

def render(include_pca: bool):
    d0 = df.copy() if include_pca else df[df["embedding"] != "pca"].copy()
    suffix = "with_pca" if include_pca else "no_pca"
    title_suffix = "PCA included" if include_pca else "PCA excluded"
    for estimator, title in [("knn", "KNN"), ("lasso", "LASSO")]:
        d = d0[d0["estimator"] == estimator].copy()
        order = d.groupby("Display name", observed=True)["L2"].mean().sort_values().index.tolist()
        fig_h = max(6.5, 0.36 * len(order) + 1.8)
        fig, ax = plt.subplots(figsize=(9.0, fig_h), constrained_layout=True)
        sns.pointplot(
            data=d, y="Display name", order=order, x="L2", hue="Model type",
            palette=palette, join=False, dodge=False, errorbar=("ci", 95),
            n_boot=5000, seed=42, markers="D", scale=0.85, errwidth=1.4,
            capsize=0.18, ax=ax,
        )
        best = d.groupby("Display name", observed=True)["L2"].mean().min()
        ax.axvline(best, color="black", linestyle="--", linewidth=1, alpha=0.55)
        ax.grid(axis="y")
        ax.set_title(f"Tahoe LFC Heldout Molecules ({title}, {title_suffix})")
        ax.set_xlabel("Mean L2 across 45 cell lines with 95% CI")
        ax.set_ylabel(None)
        ax.legend(title=None, frameon=False, loc="lower right")
        for ext in ["png", "pdf"]:
            fig.savefig(fig_dir / f"tahoe_lfc_heldout_{suffix}_{estimator}.{ext}", dpi=220, bbox_inches="tight")
        plt.show()

render(include_pca=False)
render(include_pca=True)
